# Prepare Crunchbase data for Mission Studio / Mission Radar

In [ ]:
import pandas as pd

from src import PROJECT_DIR, logging

from discovery_utils.getters import crunchbase
from discovery_utils.utils.io import safe_yaml_load
from discovery_utils.utils import (
    analysis_crunchbase,
    analysis,
    charts
)

from discovery_utils.utils.llm import batch_check


PROJECT_NAME = "2025_02_MS_asf"
OUTPUT_DIR = PROJECT_DIR / f"data/{PROJECT_NAME}"

In [ ]:
CB = crunchbase.CrunchbaseGetter()

In [ ]:
CONFIG_NAMES = [
    "bioenergy",
    "biomass_heating",
    "built_environment",
    "ccus",
    # "decarbonisation_general",
    "district_heating",
    "energy_efficiency",
    "energy_grid",
    # "energy_storage",
    "geothermal_energy",
    "green_skills",
    "heat_pumps",
    "heat_storage",
    "hydrogen_energy",
    "hydrogen_heating",
    "micro_chp",
    # "renewables_general",
    "solar_thermal",
    # "solar",
    # "wind"
]

In [ ]:
def get_config_dict(config_name: str) -> dict:
    """Find companies in a specific category from a config file"""
    config_path = str(PROJECT_DIR / f"notebooks/{PROJECT_NAME}/config_{config_name}.yaml")
    return safe_yaml_load(open(config_path))

def get_companies_from_config(config: dict) -> pd.DataFrame:
    """Get companies from a config file"""
    category_name = config["search_recipe"]["category_name"]
    return CB.get_companies_in_nesta_categories("topic_labels", [category_name])

async def check_relevance(selected_df: pd.DataFrame, config_name: str, config: dict) -> None:
    """Check relevance of the selected companies"""
    selected_texts_df = CB.get_organisation_text(selected_df)
    check_data = dict(zip(selected_texts_df['id'], selected_texts_df['text']))
    system_message = batch_check.generate_relevance_check_system_message(config)

    fields = [
        {"name": "is_relevant", "type": "str", "description": "A one-word answer: 'yes' or 'no'."},
    ]

    processor = batch_check.LLMProcessor(
        output_path=str(OUTPUT_DIR / f"llm_check_MS_{config_name}.jsonl"),
        system_message=system_message,
        session_name="mission_studio",
        output_fields=fields,
    )

    await processor.run(check_data, batch_size=10, sleep_time=0.5)    

In [ ]:
async def check_all_configs(config_names) -> None:
    """Check relevance for all config files"""
    for config_name in config_names:
        logging.info(f"Checking relevance for {config_name}")
        config = get_config_dict(config_name)
        selected_df = get_companies_from_config(config)
        await check_relevance(selected_df, config_name, config)
        

In [ ]:
await check_all_configs(CONFIG_NAMES)